In [6]:
!pip install pandas sqlalchemy mysql-connector-python scikit-learn matplotlib seaborn --quiet
print("Libraries installed successfully!")


Libraries installed successfully!


In [7]:
import pandas as pd
import numpy as np
import mysql.connector
from sqlalchemy import create_engine
import matplotlib.pyplot as plt
import seaborn as sns


In [8]:
connection = mysql.connector.connect(
    host="127.0.0.1",
    user="root",
    port="3306",
    password="omkapse@45",   
    database="eco_packaging"
)

print("Connected to MySQL successfully!")


Connected to MySQL successfully!


In [9]:
query = "SELECT * FROM materials;"
df = pd.read_sql(query, connection)
df.head()


C:\Users\Admin\AppData\Local\Temp\ipykernel_24320\242384238.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,Material_ID,Material_Type,Product_Category,Strength_MPa,Weight_Capacity_kg,Biodegradability_Score,CO2_Emission_kg,Recyclability_Percent,Source,Certification,Cost_INR,Notes
0,MAT00001,Cellulose,Textiles,22.26,28.23,37.9,11.18,67.7,Supplier A,EcoCert,3720.06,Low Stock
1,MAT00002,Recycled Paper,Electronics,55.91,24.03,68.2,8.64,71.6,Supplier D,FSC,2959.78,Low Stock
2,MAT00003,Recycled Aluminum,Household Items,406.62,1.64,84.5,10.62,34.0,Supplier C,ISO 14001,785.18,In Stock
3,MAT00004,Linen,Textiles,185.90,35.05,41.2,1.13,45.9,Supplier D,ISO 14001,663.17,Pre-Order
4,MAT00005,Recycled Glass,Coatings,443.87,36.80,35.4,1.51,66.1,Supplier A,FSC,3246.13,In Stock


In [10]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
df.describe()


Shape: (10000, 12)

Missing values:
 Material_ID               0
Material_Type             0
Product_Category          0
Strength_MPa              0
Weight_Capacity_kg        0
Biodegradability_Score    0
CO2_Emission_kg           0
Recyclability_Percent     0
Source                    0
Certification             0
Cost_INR                  0
Notes                     0
dtype: int64

Duplicate rows: 0


,Strength_MPa,Weight_Capacity_kg,Biodegradability_Score,CO2_Emission_kg,Recyclability_Percent,Cost_INR
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,252.287347,50.646350,60.001480,7.750072,50.086500,2147.630229
std,141.323221,28.733581,23.162499,4.175746,29.062067,1151.797339
min,10.050000,1.010000,20.000000,0.500000,0.000000,166.000000
25%,129.347500,25.957500,39.800000,4.120000,24.500000,1160.340000
50%,250.555000,50.525000,60.300000,7.790000,50.700000,2135.590000
75%,373.665000,75.932500,79.700000,11.350000,75.200000,3156.697500
max,500.000000,99.980000,100.000000,15.000000,100.000000,4148.340000


In [11]:
print("Shape:", df.shape)


Shape: (10000, 12)


In [12]:
print("\nMissing values:\n", df.isnull().sum())



Missing values:
 Material_ID               0
Material_Type             0
Product_Category          0
Strength_MPa              0
Weight_Capacity_kg        0
Biodegradability_Score    0
CO2_Emission_kg           0
Recyclability_Percent     0
Source                    0
Certification             0
Cost_INR                  0
Notes                     0
dtype: int64


In [13]:
print("\nDuplicate rows:", df.duplicated().sum())



Duplicate rows: 0


In [14]:
df.describe()


,Strength_MPa,Weight_Capacity_kg,Biodegradability_Score,CO2_Emission_kg,Recyclability_Percent,Cost_INR
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,252.287347,50.646350,60.001480,7.750072,50.086500,2147.630229
std,141.323221,28.733581,23.162499,4.175746,29.062067,1151.797339
min,10.050000,1.010000,20.000000,0.500000,0.000000,166.000000
25%,129.347500,25.957500,39.800000,4.120000,24.500000,1160.340000
50%,250.555000,50.525000,60.300000,7.790000,50.700000,2135.590000
75%,373.665000,75.932500,79.700000,11.350000,75.200000,3156.697500
max,500.000000,99.980000,100.000000,15.000000,100.000000,4148.340000


In [15]:
df = df.drop_duplicates().reset_index(drop=True)


In [16]:
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())


In [17]:
df['cost'] = df['cost'].astype(str).str.replace(',','').astype(float)


KeyError: 'cost'

In [ ]:
df['recyclable'] = df['recyclable'].map({True:1, False:0}).fillna(df['recyclable'])
df['recyclable'] = df['recyclable'].astype(int)


In [ ]:
df.to_csv("materials_cleaned.csv", index=False)


In [ ]:
import numpy as np

numeric_cols = ["weight", "cost", "co2_emission", "durability"]

Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("Lower bounds:\n", lower_bound)
print("\nUpper bounds:\n", upper_bound)

outliers = ((df[numeric_cols] < lower_bound) | (df[numeric_cols] > upper_bound)).sum()
print("\nOutliers per column:")
print(outliers)


Lower bounds:
 weight         -6.2625
cost            0.9500
co2_emission   -0.2950
durability      3.0000
dtype: float64

Upper bounds:
 weight          53.8375
cost            11.1500
co2_emission     1.5850
durability      11.0000
dtype: float64

Outliers per column:
weight          35
cost            11
co2_emission    27
durability       0
dtype: int64


In [ ]:

df_clean = df.copy()

df_clean[numeric_cols] = df[numeric_cols].clip(
    lower=lower_bound,
    upper=upper_bound,
    axis=1
)

# Check remaining outliers after clipping
outliers_after = (
    (df_clean[numeric_cols] < lower_bound) |
    (df_clean[numeric_cols] > upper_bound)
).sum()

print("Outliers after treatment:\n")
print(outliers_after)



Outliers after treatment:

weight          0
cost            0
co2_emission    0
durability      0
dtype: int64


In [ ]:
# Fix: clip numeric columns using column-wise bounds (axis=1)
df_clean = df.copy()

# ensure lower_bound and upper_bound are Series indexed by numeric_cols
print("lower_bound index:", list(lower_bound.index))
print("numeric_cols:", list(numeric_cols))

# Clip per-column (axis=1 aligns Series index to columns)
df_clean[numeric_cols] = df_clean[numeric_cols].clip(lower=lower_bound, upper=upper_bound, axis=1)

# Verify
outliers_after = ((df_clean[numeric_cols] < lower_bound) | (df_clean[numeric_cols] > upper_bound)).sum()
print("Outliers after clipping:\n", outliers_after)

# Quick stats to see effect
display(df[numeric_cols].describe().T.assign(before_mean=lambda x: x['mean']))
display(df_clean[numeric_cols].describe().T.assign(after_mean=lambda x: x['mean']))


lower_bound index: ['weight', 'cost', 'co2_emission', 'durability']
numeric_cols: ['weight', 'cost', 'co2_emission', 'durability']
Outliers after clipping:
 weight          0
cost            0
co2_emission    0
durability      0
dtype: int64


,count,mean,std,min,25%,50%,75%,max,before_mean
weight,500.0,26.98760,16.793005,5.00,16.275,22.50,31.300,113.40,26.98760
cost,500.0,6.26720,2.203596,1.90,4.775,6.10,7.325,18.10,6.26720
co2_emission,500.0,0.71094,0.471723,0.09,0.410,0.62,0.880,3.68,0.71094
durability,500.0,6.89200,1.237497,4.00,6.000,7.00,8.000,10.00,6.89200


,count,mean,std,min,25%,50%,75%,max,after_mean
weight,500.0,25.626825,12.677111,5.00,16.275,22.50,31.300,53.8375,25.626825
cost,500.0,6.201500,1.981091,1.90,4.775,6.10,7.325,11.1500,6.201500
co2_emission,500.0,0.675550,0.352491,0.09,0.410,0.62,0.880,1.5850,0.675550
durability,500.0,6.892000,1.237497,4.00,6.000,7.00,8.000,10.0000,6.892000


In [ ]:
# 1. CO2 Impact Index
df_clean['co2_impact_index'] = (
    df_clean['co2_emission'] / df_clean['co2_emission'].max()
)

# 2. Cost Efficiency Index
df_clean['cost_efficiency_index'] = (
    df_clean['durability'] / df_clean['cost']
)

df_clean['cost_efficiency_index'] = (
    df_clean['cost_efficiency_index'] - df_clean['cost_efficiency_index'].min()
) / (
    df_clean['cost_efficiency_index'].max() - df_clean['cost_efficiency_index'].min()
)

# 3. Sustainability Score
df_clean['sustainability_score'] = (
    0.4 * (1 - df_clean['co2_impact_index']) +
    0.3 * df_clean['recyclable'] +
    0.3 * (df_clean['durability'] / df_clean['durability'].max())
)


In [ ]:
df_clean[['co2_impact_index',
          'cost_efficiency_index',
          'sustainability_score']].describe()


,co2_impact_index,cost_efficiency_index,sustainability_score
count,500.000000,500.000000,500.000000
mean,0.426215,0.304467,0.728474
std,0.222392,0.161868,0.124435
min,0.056782,0.000000,0.120000
25%,0.258675,0.187701,0.677823
50%,0.391167,0.272058,0.755915
75%,0.555205,0.382353,0.806601
max,1.000000,1.000000,0.944763


In [ ]:
df_clean.to_csv("materials_milestone1_final.csv", index=False)


In [ ]:
pip uninstall sqlalchemy -y
pip install sqlalchemy==1.4.49
